# Using the BITS Behavior Model

This tutorial is a minimal public-API example. It builds a tiny straight-road scene directly with Tactics2D objects, constructs `BitsBehaviorModel`, and calls:

```python
trajectories = model.predict(participants, map_, frame, agent_ids)
```

No dataset parser, NuPlan database, WOMD file, explicit checkpoint path, torch wrapper, or custom policy is needed in this notebook.

## 1. Setup

The only external requirement for this demo is the optional BITS runtime dependencies. The default weighted policy is loaded by `BitsBehaviorModel()`; the scene data below is created in memory.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from shapely.geometry import LineString, Polygon


def find_repo_root(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "tactics2d" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Cannot find the Tactics2D repository root.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from tactics2d.behavior import BitsBehaviorModel
from tactics2d.map.element import Area, Lane, Map, RoadLine
from tactics2d.participant.element import Vehicle
from tactics2d.participant.trajectory import State, Trajectory

try:
    import torch
except ImportError:
    torch = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") if torch is not None else "cpu"
print("repo_root:", repo_root)
print("python:", sys.executable)
print("torch installed:", torch is not None)
print("device:", DEVICE)

## 2. Construct The Behavior Model

`BitsBehaviorModel()` loads the packaged BITS policy and its matching saved config. The config is not tuned in this tutorial because its history length, raster channels, and planning horizon must match the trained weights.

In [ ]:
if torch is None:
    raise ImportError(
        "torch is required for the default BITS policy. "
        f"Current Python is {sys.executable!r}; select a kernel with tactics2d[bits] installed."
    )

model = BitsBehaviorModel(device=DEVICE)
config = model.config

print("history steps:", config.history_steps)
print("planning steps:", config.planning_steps)
print("BITS future_steps:", config.future_steps)
print("dt:", config.dt)
print("raster size:", config.raster_size)
print("max agents:", config.max_agents)

## 3. Build A Minimal Road Map

The default BITS model expects a rasterized map. For a self-contained demo we create one straight drivable lane and a surrounding drivable area directly in Tactics2D.

In [ ]:
def build_straight_road(length):
    road = Map(name="bits_minimal_straight_road", scenario_type="demo")
    left = LineString([(-20.0, 1.8), (length, 1.8)])
    right = LineString([(-20.0, -1.8), (length, -1.8)])
    lane = Lane(
        "lane_0",
        left_side=left,
        right_side=right,
        custom_tags={"centerline": [(-20.0, 0.0), (length, 0.0)]},
    )
    drivable_area = Area(
        "road_surface",
        Polygon([(-20.0, -4.0), (length, -4.0), (length, 4.0), (-20.0, 4.0)]),
        subtype="drivable_area",
    )
    centerline = RoadLine(
        "lane_centerline",
        LineString([(-20.0, 0.0), (length, 0.0)]),
        subtype="dashed",
        width=0.15,
    )
    road.add_lane(lane)
    road.add_area(drivable_area)
    road.add_roadline(centerline)
    return road


EGO_ID = "ego"
NEIGHBOR_ID = "lead_vehicle"
CONTROLLED_IDS = [EGO_ID]
PREDICTION_FRAME = config.history_steps * config.step_ms
CLOSED_LOOP_STEPS = 5
ROAD_LENGTH = 8.0 * (config.history_steps + config.planning_steps + CLOSED_LOOP_STEPS) * config.dt + 80.0
map_ = build_straight_road(ROAD_LENGTH)

print("prediction frame:", PREDICTION_FRAME)
print("map lanes:", len(map_.lanes))
print("map areas:", len(map_.areas))
print("map roadlines:", len(map_.roadlines))


## 4. Build Participants In Memory

Both behavior-model demos use the normal Tactics2D scene format: `participants` is a dictionary of `Vehicle` objects, each vehicle owns a `Trajectory`, and each trajectory contains time-indexed `State` objects.

For BITS, the default learned policy consumes a fixed history window from the saved config. With the default weights used here, each controlled agent should have `history_steps + 1` states ending at `PREDICTION_FRAME`, sampled every `step_ms`. Each state should provide `frame`, `x`, `y`, `heading`, and either `vx`/`vy` or `speed`; vehicle `length` and `width` are used for rasterization and interaction costs. The model returns `planning_steps` future states (`future_steps` in the BITS checkpoint config).

This live scene stops at the current prediction frame so future states can be appended during closed-loop rollout.


In [ ]:
def make_vehicle(id_, x0, y0, speed, last_frame, config):
    trajectory = Trajectory(id_, fps=round(1.0 / config.dt, 3))
    last_step = last_frame // config.step_ms
    for step in range(last_step + 1):
        frame = step * config.step_ms
        t = step * config.dt
        trajectory.add_state(
            State(
                frame=frame,
                x=x0 + speed * t,
                y=y0,
                heading=0.0,
                vx=speed,
                vy=0.0,
                speed=speed,
            )
        )
    return Vehicle(
        id_,
        trajectory=trajectory,
        length=4.8,
        width=2.0,
        verify=False,
    )


participants = {
    EGO_ID: make_vehicle(EGO_ID, x0=0.0, y0=0.0, speed=6.0, last_frame=PREDICTION_FRAME, config=config),
    NEIGHBOR_ID: make_vehicle(
        NEIGHBOR_ID,
        x0=18.0,
        y0=0.0,
        speed=5.0,
        last_frame=PREDICTION_FRAME,
        config=config,
    ),
}

ego = participants[EGO_ID]
current = ego.trajectory.get_state(PREDICTION_FRAME)

print("participants:", list(participants.keys()))
print("required history states:", config.history_steps + 1)
print("future states returned:", config.planning_steps)
print("ego history frame count:", len(ego.trajectory.frames))
print("ego history frames:", ego.trajectory.frames[:3], "...", ego.trajectory.frames[-3:])
print("ego current xy:", [round(value, 3) for value in current.location])


## 5. Predict Trajectories

The public call is one line: pass participants, map, frame, and target agent ids. The returned value is a dictionary of Tactics2D `Trajectory` objects in world coordinates.

In [ ]:
predicted = model.predict(
    participants=participants,
    map_=map_,
    frame=PREDICTION_FRAME,
    agent_ids=[EGO_ID],
)

trajectory = predicted[EGO_ID]
print("predicted state count:", len(trajectory.frames))
print("predicted frames:", trajectory.frames[:5], "...", trajectory.frames[-1])

for state_frame in trajectory.frames[:5]:
    state = trajectory.get_state(state_frame)
    print(" ", state_frame, round(state.x, 3), round(state.y, 3), round(state.heading, 3))

## 6. Execute One Closed-Loop Update

After a single `predict(...)` call, a closed-loop simulation usually commits only the first future state. The participant is updated to that state, and the model can be called again at the new frame.


In [ ]:
ego_plan = predicted[EGO_ID]
next_frame = ego_plan.frames[0]
next_state = ego_plan.get_state(next_frame)

live_vehicle = make_vehicle(EGO_ID, x0=0.0, y0=0.0, speed=6.0, last_frame=PREDICTION_FRAME, config=config)
live_vehicle.add_state(next_state)

print("executed frame:", live_vehicle.current_state.frame)
print("executed location:", tuple(round(value, 3) for value in live_vehicle.current_state.location))
print("available future frames from this plan:", ego_plan.frames[:3], "...", ego_plan.frames[-3:])


## 7. Run A Short Closed-Loop Rollout

The loop below repeats the same receding-horizon pattern for a few control updates. At each update, the model predicts from the participant's latest state, the simulation commits only the first future state, and the next iteration replans from that updated state.


In [ ]:
closed_loop_participants = {
    EGO_ID: make_vehicle(EGO_ID, x0=0.0, y0=0.0, speed=6.0, last_frame=PREDICTION_FRAME, config=config),
    NEIGHBOR_ID: make_vehicle(
        NEIGHBOR_ID,
        x0=18.0,
        y0=0.0,
        speed=5.0,
        last_frame=PREDICTION_FRAME,
        config=config,
    ),
}
controlled_ids = CONTROLLED_IDS
current_frame = PREDICTION_FRAME
executed_trace = [closed_loop_participants[EGO_ID].current_state.location]

for step in range(CLOSED_LOOP_STEPS):
    step_predictions = model.predict(
        closed_loop_participants,
        map_,
        frame=current_frame,
        agent_ids=controlled_ids,
    )
    ego_step_plan = step_predictions[EGO_ID]
    next_frame = ego_step_plan.frames[0]
    next_state = ego_step_plan.get_state(next_frame)
    closed_loop_participants[EGO_ID].add_state(next_state)
    current_frame = next_state.frame
    executed_trace.append(next_state.location)
    print(
        f"step {step + 1}",
        "frame:", current_frame,
        "location:", tuple(round(value, 3) for value in next_state.location),
    )

print("executed trace:", [tuple(round(value, 3) for value in point) for point in executed_trace])


## 8. Use Parsed Dataset Data

The same public API works when participants and maps come from a dataset parser. This cell is intentionally not executed here because NuPlan files and map paths are environment-specific.


In [ ]:
# from tactics2d.behavior import BitsBehaviorModel
# from tactics2d.dataset_parser import NuPlanParser
#
# parser = NuPlanParser()
# participants, _ = parser.parse_trajectory(
#     data_file,
#     data_folder,
#     time_range=time_range,
# )
# map_ = parser.parse_map(map_file, map_folder)
#
# model = BitsBehaviorModel(device=device)
# predicted = model.predict(
#     participants=participants,
#     map_=map_,
#     frame=frame,
#     agent_ids=agent_ids,
# )


## 9. What Stays Internal

Checkpoint paths, torch modules, policy wrappers, candidate goal sampling, scoring details, and reproduction metrics are intentionally not part of this public tutorial. The advanced reproduction notebook covers those internals for users who want to inspect or reproduce the BITS training/evaluation path.
